# Lab 1 — PCA, Clustering y Monitoreo Estructural (SHM)

**Sesión:** reducción de dimensionalidad y agrupamiento · **Tema:** PCA, KMeans (codo), DBSCAN y clasificación

**Público:** ingenieros civiles · **Entorno:** GitHub Codespaces o `labs/.venv`

## Cómo trabajar

1. Abre este repo en **Codespaces** o ejecuta `bash labs/setup.sh` y activa `source labs/.venv/bin/activate`.
2. Lee [`data/DATOS.md`](data/DATOS.md) para entender sensores y etiquetas de condición.
3. Ejecuta celdas **en orden**; solo modifica bloques `### TU TAREA AQUÍ ###`.
4. Tras cada pregunta, ejecuta **Autoevaluación** y busca ✅.
5. Referencia docente: `pca_monitoreo_estructural_solucion.ipynb`.

> El código de carga, limpieza, gráficos y modelado está pre-escrito. Tu trabajo: **experimentar** con parámetros y **interpretar** resultados de monitoreo estructural.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_contexto_pca, verificar_carga, verificar_limpieza, verificar_columna,
    verificar_resumen, verificar_etiquetas, verificar_correlacion, verificar_escalado,
    verificar_varianza, verificar_proyeccion_2d, verificar_loadings,
    verificar_clasificacion_pca, verificar_kmeans, verificar_dbscan,
    verificar_comparativa_clustering, verificar, resumen_seccion,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, silhouette_score, adjusted_rand_score

%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Entorno listo (Codespaces / labs/.venv).')


## Contexto del dataset (Kaggle SHM)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Accel_X, Accel_Y, Accel_Z | m/s² | Vibración y movimiento en tres ejes |
| Strain | με | Deformación del material (extensómetro) |
| Temp | °C | Temperatura ambiente / del sensor |
| **Condition Label** | **0 / 1 / 2** | **Estado estructural (normal / daño menor / severo)** |

Fuente: Ziya07 · 1 000 lecturas · PCA **no supervisado** (no usa la etiqueta para crear componentes).

Detalle ampliado: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Contexto PCA en monitoreo estructural

### 📘 Subpreguntas
- **1.a** ¿PCA es un método **supervisado** o **no supervisado**?
- **1.b** ¿Por qué es útil reducir dimensionalidad antes de visualizar o clasificar lecturas de sensores?

En obra, decenas de sensores generan datos redundantes; PCA condensa esa información en pocas componentes interpretables.


#### ✍️ Tu respuesta (opcional, 2–3 líneas)




In [ ]:
### TU TAREA AQUÍ ###
# Indica el método de reducción de dimensionalidad para este lab.
METODO_REDUCCION = "pca"  # Prueba "tsne" o "umap" y revisa la autoevaluación

print(f"Método elegido: {METODO_REDUCCION}")


In [ ]:
# --- Autoevaluación 1 ---
r = verificar_contexto_pca(METODO_REDUCCION)
resumen_seccion('1 — Contexto PCA', r)


## Pregunta 2 — Carga del dataset de sensores

### 📘 Subpreguntas
- **2.a** ¿Cuántas **features** de sensor hay frente a 1 **etiqueta** de condición?
- **2.b** ¿Por qué incluimos `Timestamp` pero no la usamos en PCA?


In [ ]:
# --- PRE-ESCRITO: carga desde data/building_health_monitoring_dataset.csv ---
RUTA_DATOS = Path("data/building_health_monitoring_dataset.csv")
df = pd.read_csv(RUTA_DATOS)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} filas × {df.shape[1]} columnas")


In [ ]:
### TU TAREA AQUÍ ###
FEATURES = [
    "Accel_X (m/s^2)",
    "Accel_Y (m/s^2)",
    "Accel_Z (m/s^2)",
    "Strain (με)",
    "Temp (°C)",
]  # Quita o agrega columnas y observa la autoevaluación

N_FILAS_HEAD = 5  # Prueba 3, 8 o 10

print(f"Features para PCA: {FEATURES}")
print(f"Primeras {N_FILAS_HEAD} lecturas:")
display(df.head(N_FILAS_HEAD))


In [ ]:
# --- Autoevaluación 2 ---
r = verificar_carga(df, N_FILAS_HEAD)
r.append(verificar(len(FEATURES) == 5, '✅ 5 features de sensores definidas.', '❌ FEATURES debe incluir las 5 columnas de sensores.'))
resumen_seccion('2 — Carga', r)
print("ℹ️ 5 features + Timestamp + Condition Label.")


## Pregunta 3 — Calidad de datos y limpieza

### 📘 Subpreguntas
- **3.a** ¿Cuántas filas se pierden al eliminar lecturas con nulos?
- **3.b** ¿Qué sensor revisarías primero y por qué?


In [ ]:
# --- PRE-ESCRITO: conteo de nulos y limpieza ---
n_antes = len(df)
n_nulos_por_col = df[FEATURES].isna().sum()
print("Nulos por sensor (datos crudos):")
display(n_nulos_por_col)

df_limpio = df.dropna(subset=FEATURES).copy()
n_despues = len(df_limpio)
print(f"Tras dropna: {n_antes} → {n_despues} lecturas válidas")


In [ ]:
### TU TAREA AQUÍ ###
COLUMNA_REVISAR = "Strain (με)"  # Prueba "Accel_Z (m/s^2)" o "Temp (°C)"

stats_col = df[COLUMNA_REVISAR].describe()
print(f"Estadísticas de «{COLUMNA_REVISAR}» (datos crudos):")
display(stats_col)


In [ ]:
# --- Autoevaluación 3 ---
r = verificar_limpieza(df_limpio, n_antes, n_despues)
r.extend(verificar_columna(df, COLUMNA_REVISAR))
resumen_seccion('3 — Calidad', r)


## Pregunta 4 — Estadísticas descriptivas de sensores

### 📘 Subpreguntas
- **4.a** Mirando `describe()`, ¿qué sensor tiene mayor dispersión relativa (std vs |media|)?
- **4.b** ¿Por qué esto importa antes de aplicar PCA?


In [ ]:
### TU TAREA AQUÍ ###
COLUMNAS_RESUMEN = ["Strain (με)", "Temp (°C)", "Accel_Z (m/s^2)"]  # Quita o agrega columnas

resumen = df_limpio[COLUMNAS_RESUMEN].describe()
display(resumen)

medias_abs = df_limpio[COLUMNAS_RESUMEN].abs().mean()
dispersion = (df_limpio[COLUMNAS_RESUMEN].std() / medias_abs).sort_values(ascending=False)
print("\nDispersión relativa (std / |media|):")
display(dispersion)


In [ ]:
# --- Autoevaluación 4 ---
r = verificar_resumen(resumen, COLUMNAS_RESUMEN)
resumen_seccion('4 — Describe', r)


## Pregunta 5 — Distribución de Condition Label

### 📘 Subpreguntas
- **5.a** ¿Qué clase es mayoritaria (0, 1 o 2)?
- **5.b** ¿Por qué es típico tener más lecturas «normales» en SHM?


In [ ]:
# --- PRE-ESCRITO: histograma de etiquetas ---
conteo = df_limpio['Condition Label'].value_counts().sort_index().to_dict()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(conteo.keys(), conteo.values(), color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
ax.set_xlabel('Condition Label')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de estados estructurales')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['0 — Normal', '1 — Daño menor', '2 — Daño severo'])
plt.tight_layout()
plt.show()


In [ ]:
### TU TAREA AQUÍ ###
N_CLASES_MOSTRAR = 3  # Prueba 1 o 2

serie_clases = pd.Series(conteo).sort_index().head(N_CLASES_MOSTRAR)
print(f"Distribución (top {N_CLASES_MOSTRAR} clases):")
display(serie_clases)


In [ ]:
# --- Autoevaluación 5 ---
r = verificar_etiquetas(conteo, N_CLASES_MOSTRAR)
resumen_seccion('5 — Etiquetas', r)


## Pregunta 6 — Correlación entre sensores

### 📘 Subpreguntas
- **6.a** ¿Qué par de sensores está **más correlacionado** (|r|)?
- **6.b** ¿Qué implica alta correlación para PCA?


In [ ]:
# --- PRE-ESCRITO: matriz de correlación ---
corr = df_limpio[FEATURES].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlación entre sensores (datos limpios)')
plt.tight_layout()
plt.show()


In [ ]:
### TU TAREA AQUÍ ###
TOP_N_PARES = 3  # Prueba 2 o 5

pares = []
for i, c1 in enumerate(FEATURES):
    for c2 in FEATURES[i + 1:]:
        pares.append((c1, c2, abs(corr.loc[c1, c2])))
pares_ordenados = sorted(pares, key=lambda x: x[2], reverse=True)
top_pares = pares_ordenados[:TOP_N_PARES]
max_corr = top_pares[0][2]
top_par = (top_pares[0][0], top_pares[0][1])

print(f"Correlación máxima (|r|) = {max_corr:.3f}")
print(f"Top {TOP_N_PARES} pares:")
for c1, c2, r_val in top_pares:
    print(f"  {c1} ↔ {c2}: |r| = {r_val:.3f}")


In [ ]:
# --- Autoevaluación 6 ---
r = verificar_correlacion(max_corr, top_par)
resumen_seccion('6 — Correlación', r)


## Pregunta 7 — Estandarización antes de PCA

### 📘 Subpreguntas
- **7.a** ¿Por qué `StandardScaler` es recomendable antes de PCA?
- **7.b** ¿Qué sensor domina PC1 si **no** escalas (pista: gravedad en Accel_Z)?


In [ ]:
# --- PRE-ESCRITO: comparar varianza PC1 con y sin escalado ---
X_raw = df_limpio[FEATURES].values
scaler_ref = StandardScaler()
X_scaled_ref = scaler_ref.fit_transform(X_raw)

pca_crudo = PCA(n_components=5, random_state=42)
pca_crudo.fit(X_raw)
var_pc1_crudo = float(pca_crudo.explained_variance_ratio_[0])

pca_esc = PCA(n_components=5, random_state=42)
pca_esc.fit(X_scaled_ref)
var_pc1_escalado = float(pca_esc.explained_variance_ratio_[0])

print(f"PC1 sin escalado:  {var_pc1_crudo*100:.1f}% de varianza")
print(f"PC1 con escalado:  {var_pc1_escalado*100:.1f}% de varianza")


In [ ]:
### TU TAREA AQUÍ ###
ESCALAR = True  # Prueba False y compara varianza PC1

if ESCALAR:
    scaler = StandardScaler()
    X_pca_input = scaler.fit_transform(X_raw)
else:
    X_pca_input = X_raw

pca_tmp = PCA(n_components=5, random_state=42)
pca_tmp.fit(X_pca_input)
var_pc1_actual = float(pca_tmp.explained_variance_ratio_[0])
print(f"ESCALAR={ESCALAR} → PC1 explica {var_pc1_actual*100:.1f}% de varianza")


In [ ]:
# --- Autoevaluación 7 ---
r = verificar_escalado(ESCALAR, var_pc1_escalado, var_pc1_crudo)
resumen_seccion('7 — Escalado', r)


## Pregunta 8 — Varianza explicada (scree plot)

### 📘 Subpreguntas
- **8.a** ¿Cuántas componentes necesitas para capturar el **90%** de varianza?
- **8.b** ¿Qué trade-off hay al usar menos componentes?


In [ ]:
# --- PRE-ESCRITO: PCA completo y scree plot ---
pca_full = PCA(random_state=42)
pca_full.fit(X_pca_input)
var_ratio = pca_full.explained_variance_ratio_
var_acum = np.cumsum(var_ratio).tolist()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(var_ratio) + 1), var_ratio, color='#3498db', edgecolor='white')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Varianza explicada')
axes[0].set_title('Scree plot — varianza por componente')

axes[1].plot(range(1, len(var_acum) + 1), var_acum, 'o-', color='#e74c3c')
axes[1].axhline(0.90, color='gray', linestyle='--', label='90%')
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Varianza acumulada')
axes[1].set_title('Varianza acumulada')
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
### TU TAREA AQUÍ ###
UMBRAL_VARIANZA = 0.90  # Prueba 0.80 o 0.95
N_COMPONENTES = 5       # Ajusta según el umbral elegido

n_min = int(np.searchsorted(var_acum, UMBRAL_VARIANZA) + 1)
print(f"Umbral {UMBRAL_VARIANZA:.0%} → mínimo {n_min} componente(s)")
print(f"Tu N_COMPONENTES = {N_COMPONENTES}")
print("Varianza acumulada:", [f"{v:.3f}" for v in var_acum])


In [ ]:
# --- Autoevaluación 8 ---
r = verificar_varianza(var_acum, UMBRAL_VARIANZA, N_COMPONENTES)
resumen_seccion('8 — Varianza', r)


## Pregunta 9 — Proyección 2D: PCA lineal y Kernel PCA

### 📘 Subpreguntas
- **9.a** ¿Las clases de daño se **separan** visualmente en el plano PC1–PC2 (PCA lineal)?
- **9.b** ¿Por qué coloreamos por `Condition Label` si PCA no la usó?
- **9.c** ¿Mejora un **Kernel PCA** no lineal la separación y el clustering posterior?
- **9.d** ¿Qué kernel (`linear`, `poly`, `rbf`, `cosine`, `sigmoid`) funciona mejor en este dataset?


In [ ]:
# --- PRE-ESCRITO: transformación a componentes principales ---
X_pca = pca_full.transform(X_pca_input)
var_pc1 = float(var_ratio[0])
var_pc2 = float(var_ratio[1])
print(f"Proyección lista: {X_pca.shape[0]} puntos × {X_pca.shape[1]} componentes")


In [ ]:
### TU TAREA AQUÍ ###
COLOREAR_POR = "Condition Label"
ETIQUETAS = {0: "0 — Normal", 1: "1 — Daño menor", 2: "2 — Daño severo"}
COLORES = {0: "#2ecc71", 1: "#f39c12", 2: "#e74c3c"}

fig, ax = plt.subplots(figsize=(8, 6))
for label, nombre in ETIQUETAS.items():
    mask = df_limpio[COLOREAR_POR] == label
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=COLORES[label], label=nombre, alpha=0.75, edgecolors="white", linewidth=0.3, s=45,
    )
ax.set_xlabel(f"PC1 ({var_pc1*100:.1f}% varianza)")
ax.set_ylabel(f"PC2 ({var_pc2*100:.1f}% varianza)")
ax.set_title("Proyección PCA lineal — estados estructurales")
ax.legend(title=COLOREAR_POR, loc="best")
plt.tight_layout()
plt.show()
n_muestras = len(X_pca)


In [ ]:
# --- Autoevaluación 9 ---
r = verificar_proyeccion_2d(n_muestras, COLOREAR_POR, var_pc1, var_pc2)
resumen_seccion('9 — Proyección 2D', r)


### 9.c — Kernel PCA (subsección)

- **Kernel PCA** captura **interacciones no lineales** entre sensores (sigue siendo no supervisado).
- Usaremos **3 componentes kernel** para KMeans y DBSCAN (`X_cluster_input`) y visualizaremos **KPC1 vs KPC2**.
- Parámetros de partida: kernel `poly`, `degree=4`, `gamma=0.2`, `coef0=1.0`.


In [ ]:
### TU TAREA AQUÍ ###
KERNEL = "poly"          # Prueba: "linear", "rbf", "sigmoid", "cosine"
KERNEL_DEGREE = 4        # Solo para kernel "poly"
KERNEL_GAMMA = 0.2       # Para poly / rbf / sigmoid
KERNEL_COEF0 = 1.0       # Solo para poly / sigmoid
N_COMPONENTES_KPCA = 3   # 3D para clustering; visualizamos las 2 primeras

kpca = KernelPCA(
    n_components=N_COMPONENTES_KPCA,
    kernel=KERNEL,
    degree=KERNEL_DEGREE,
    gamma=KERNEL_GAMMA,
    coef0=KERNEL_COEF0,
    random_state=42,
)
X_kpca = kpca.fit_transform(X_pca_input)
X_cluster_input = X_kpca  # KMeans y DBSCAN usan este espacio

y_color = df_limpio[COLOREAR_POR].values
sil_linear = silhouette_score(X_pca[:, :2], y_color)
sil_kernel = silhouette_score(X_kpca[:, :2], y_color)
print(f"Silhouette PCA lineal (PC1–PC2):    {sil_linear:.3f}")
print(f"Silhouette Kernel PCA (KPC1–KPC2): {sil_kernel:.3f}")
print(f"Espacio de clustering: {X_cluster_input.shape[1]} componentes kernel")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, coords, xlab, ylab, title in [
    (axes[0], X_pca[:, :2], f"PC1 ({var_pc1*100:.1f}% var.)", f"PC2 ({var_pc2*100:.1f}% var.)", "PCA lineal"),
    (axes[1], X_kpca[:, :2], "KPC1", "KPC2", f"Kernel PCA ({KERNEL})"),
]:
    for label, nombre in ETIQUETAS.items():
        mask = y_color == label
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c=COLORES[label], label=nombre, alpha=0.75, edgecolors="white", linewidth=0.3, s=40,
        )
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.set_title(title)
    ax.legend(title=COLOREAR_POR, loc="best", fontsize=8)
plt.suptitle("9.c — Comparación PCA lineal vs Kernel PCA", y=1.02)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(x=X_pca[:, 0], hue=df_limpio[COLOREAR_POR], ax=axes[0], common_norm=False, palette=COLORES)
axes[0].set_title("PC1 por Condition Label")
sns.kdeplot(x=X_kpca[:, 0], hue=df_limpio[COLOREAR_POR], ax=axes[1], common_norm=False, palette=COLORES)
axes[1].set_title("KPC1 por Condition Label")
plt.tight_layout()
plt.show()


### 9.d — Comparación de kernels

Tabla y gráficos de distintos kernels en `sklearn` (`linear`, `poly`, `rbf`, `cosine`, `sigmoid`). La métrica Silhouette vs `Condition Label` solo **evalúa** la proyección (no entrena con la etiqueta).


In [ ]:
# --- PRE-ESCRITO: comparación de kernels (9.d) ---
KERNEL_CANDIDATES = [
    {"name": "linear",  "kernel": "linear",  "params": {}},
    {"name": "poly-2",  "kernel": "poly",    "params": {"degree": 2, "gamma": 0.2, "coef0": 1.0}},
    {"name": "poly-3",  "kernel": "poly",    "params": {"degree": 3, "gamma": 0.1, "coef0": 1.0}},
    {"name": "poly-4",  "kernel": "poly",    "params": {"degree": 4, "gamma": 0.2, "coef0": 1.0}},
    {"name": "rbf-0.05","kernel": "rbf",     "params": {"gamma": 0.05}},
    {"name": "rbf-0.1", "kernel": "rbf",     "params": {"gamma": 0.1}},
    {"name": "sigmoid", "kernel": "sigmoid", "params": {"gamma": 0.1, "coef0": 1.0}},
    {"name": "cosine",  "kernel": "cosine",  "params": {}},
]

filas = []
proyecciones = {}
for cfg in KERNEL_CANDIDATES:
    kpca_cmp = KernelPCA(
        n_components=3,
        kernel=cfg["kernel"],
        random_state=42,
        **cfg["params"],
    )
    Xk = kpca_cmp.fit_transform(X_pca_input)
    sil_2d = silhouette_score(Xk[:, :2], df_limpio["Condition Label"])
    sil_3d = silhouette_score(Xk, df_limpio["Condition Label"])
    filas.append({"kernel": cfg["name"], "silhouette_2d": sil_2d, "silhouette_3d": sil_3d})
    proyecciones[cfg["name"]] = Xk

tabla_kernels = pd.DataFrame(filas).sort_values("silhouette_2d", ascending=False)
display(tabla_kernels.round(3))


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for ax, cfg in zip(axes, KERNEL_CANDIDATES):
    Xk = proyecciones[cfg["name"]]
    for label, nombre in ETIQUETAS.items():
        mask = df_limpio[COLOREAR_POR] == label
        ax.scatter(Xk[mask, 0], Xk[mask, 1], c=COLORES[label], s=25, alpha=0.7)
    ax.set_title(cfg["name"])
    ax.set_xlabel("KPC1")
    ax.set_ylabel("KPC2")
plt.suptitle("Kernel PCA — comparación de kernels (KPC1 vs KPC2)")
plt.tight_layout()
plt.show()


## Pregunta 10 — KMeans y método del codo (elbow)

### 📘 Subpreguntas
- **10.a** ¿Por qué conviene escalar features **antes** de KMeans?
- **10.b** ¿Qué valor de **k** sugiere el gráfico del codo para este edificio instrumentado?

KMeans se aplica sobre **`X_cluster_input`** (espacio Kernel PCA de 9.c), sin usar `Condition Label`.


In [ ]:
# --- PRE-ESCRITO: inercia vs k (espacio Kernel PCA) ---
K_MIN, K_MAX = 2, 10
inertias = []
for k in range(K_MIN, K_MAX + 1):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_input)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(K_MIN, K_MAX + 1), inertias, "o-", color="#2980b9", linewidth=2)
ax.set_xlabel("Número de clústeres k")
ax.set_ylabel("Inercia (within-cluster SS)")
ax.set_title("Método del codo — KMeans sobre Kernel PCA (9.c)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
### TU TAREA AQUÍ ###
K_OPT = 3  # Ajusta según el codo (prueba 2, 4 o 5)

kmeans = KMeans(n_clusters=K_OPT, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X_cluster_input)
sil_km = silhouette_score(X_cluster_input, labels_km)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_kpca[:, 0], X_kpca[:, 1], c=labels_km, cmap="tab10", alpha=0.7)
ax.set_xlabel("KPC1")
ax.set_ylabel("KPC2")
ax.set_title(f"KMeans k={K_OPT} — agrupación en plano Kernel PCA")
plt.colorbar(sc, ax=ax, label="Clúster")
plt.tight_layout()
plt.show()
print(f"Silhouette KMeans (Kernel PCA) = {sil_km:.3f}")


In [ ]:
# --- Autoevaluación 10 ---
r = verificar_kmeans(K_OPT, K_MIN, K_MAX, sil_km)
resumen_seccion('10 — KMeans y codo', r)


## Pregunta 11 — DBSCAN (densidad y ruido)

### 📘 Subpreguntas
- **11.a** ¿Qué significa la etiqueta **-1** en DBSCAN?
- **11.b** ¿Cómo influyen `eps` y `min_samples` en el número de clústeres y puntos ruidosos?

DBSCAN también usa **`X_cluster_input`** (Kernel PCA). Visualizamos en el plano KPC1–KPC2.


In [ ]:
# --- PRE-ESCRITO: sensibilidad rápida a eps en Kernel PCA ---
for eps_demo in (0.5, 0.7, 1.0):
    db_demo = DBSCAN(eps=eps_demo, min_samples=8).fit(X_cluster_input)
    n_cl = len(set(db_demo.labels_)) - (1 if -1 in db_demo.labels_ else 0)
    n_noise = int((db_demo.labels_ == -1).sum())
    print(f"eps={eps_demo:.1f} → {n_cl} clúster(es), {n_noise} puntos ruido (-1)")


In [ ]:
### TU TAREA AQUÍ ###
EPS = 0.7          # Radio de vecindad (prueba 0.5, 0.9 o 1.0)
MIN_SAMPLES = 8    # Mínimo de puntos para formar núcleo denso

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
labels_db = dbscan.fit_predict(X_cluster_input)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = int((labels_db == -1).sum())
mask_db = labels_db != -1
sil_db = silhouette_score(X_cluster_input[mask_db], labels_db[mask_db])

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_kpca[:, 0], X_kpca[:, 1], c=labels_db, cmap="coolwarm", alpha=0.7)
ax.set_xlabel("KPC1")
ax.set_ylabel("KPC2")
ax.set_title(f"DBSCAN eps={EPS}, min_samples={MIN_SAMPLES}")
plt.colorbar(sc, ax=ax, label="Clúster (-1 = ruido)")
plt.tight_layout()
plt.show()
print(f"Clústeres={n_clusters_db}, ruido={n_noise_db}, silhouette={sil_db:.3f}")


In [ ]:
# --- Autoevaluación 11 ---
r = verificar_dbscan(EPS, MIN_SAMPLES, n_clusters_db, n_noise_db, sil_db)
resumen_seccion('11 — DBSCAN', r)


## Pregunta 12 — Comparativa KMeans vs DBSCAN vs etiqueta real

### 📘 Subpreguntas
- **12.a** ¿Qué mide el **índice de Silhouette** (más alto = mejor separación)?
- **12.b** ¿Qué mide el **Adjusted Rand Index (ARI)** frente a `Condition Label`?
- **12.c** ¿Cuál método se alinea mejor con los estados estructurales conocidos?


In [ ]:
# --- PRE-ESCRITO: etiqueta de referencia (supervisada, solo para comparar) ---
y_true = df_limpio['Condition Label'].values
ari_km = adjusted_rand_score(y_true, labels_km)
ari_db = adjusted_rand_score(y_true[mask_db], labels_db[mask_db]) if mask_db.sum() > 0 else float('nan')
print("Etiquetas reales: 0=normal, 1=daño menor, 2=severo")


In [ ]:
### TU TAREA AQUÍ ###
METRICA_PRIORITARIA = 'silhouette'  # Prueba 'ari' y compara resultados
comparativa = pd.DataFrame({
    'Método': ['KMeans', 'DBSCAN'],
    'k / clústeres': [K_OPT, n_clusters_db],
    'Silhouette': [sil_km, sil_db],
    'ARI vs Condition Label': [ari_km, ari_db],
    'Puntos ruido (-1)': [0, n_noise_db],
})
display(comparativa.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(2)
axes[0].bar(x, comparativa['Silhouette'], color=['#3498db', '#e67e22'])
axes[0].set_xticks(x); axes[0].set_xticklabels(comparativa['Método'])
axes[0].set_title('Silhouette (espacio Kernel PCA)')
axes[1].bar(x, comparativa['ARI vs Condition Label'], color=['#3498db', '#e67e22'])
axes[1].set_xticks(x); axes[1].set_xticklabels(comparativa['Método'])
axes[1].set_title('ARI vs daño real')
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 12 ---
r = verificar_comparativa_clustering(sil_km, sil_db, ari_km, ari_db, METRICA_PRIORITARIA)
resumen_seccion('12 — Comparativa clustering', r)


## Pregunta 13 — Loadings y clasificación (original vs PCA)

### 📘 Subpreguntas
- **13.a** ¿Qué sensor tiene mayor peso (loading) en PC1?
- **13.b** ¿La clasificación con 3 componentes PCA pierde mucho accuracy frente a features originales?


In [ ]:
# --- PRE-ESCRITO: matriz de loadings ---
loadings = pd.DataFrame(
    pca_full.components_.T,
    index=FEATURES,
    columns=[f'PC{i+1}' for i in range(len(FEATURES))],
)
feature_pc1 = loadings['PC1'].abs().idxmax()

fig, ax = plt.subplots(figsize=(8, 4))
loadings['PC1'].sort_values().plot(kind='barh', ax=ax, color='#9b59b6')
ax.set_title(f'Loadings PC1 — mayor peso: {feature_pc1}')
ax.set_xlabel('Peso en PC1')
plt.tight_layout()
plt.show()
display(loadings.round(3))


In [ ]:
### TU TAREA AQUÍ ###
N_COMPONENTES_ML = 3  # Prueba 2 o 4
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_ml = StandardScaler().fit_transform(df_limpio[FEATURES])
y_ml = df_limpio['Condition Label']

X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_ml, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_ml,
)

rf_orig = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_orig.fit(X_train, y_train)
acc_orig = accuracy_score(y_test, rf_orig.predict(X_test))

pca_ml = PCA(n_components=N_COMPONENTES_ML, random_state=RANDOM_STATE)
X_train_pca = pca_ml.fit_transform(X_train)
X_test_pca = pca_ml.transform(X_test)

rf_pca = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_pca.fit(X_train_pca, y_train)
acc_pca = accuracy_score(y_test, rf_pca.predict(X_test_pca))

print(f"Accuracy features originales: {acc_orig:.3f}")
print(f"Accuracy PCA ({N_COMPONENTES_ML} comp.): {acc_pca:.3f}")
print(f"Diferencia Δacc = {acc_orig - acc_pca:.3f}")


In [ ]:
# --- Autoevaluación 13 ---
r = verificar_loadings(feature_pc1)
r.extend(verificar_clasificacion_pca(acc_orig, acc_pca, N_COMPONENTES_ML))
resumen_seccion('13 — Loadings y clasificación', r)


## Cierre profesional

### ✍️ Reflexión final (3 frases)
- El sensor que más peso tuvo en PC1 fue ___ porque ___.
- Para monitoreo en obra usaría PCA para ___.
- Antes de activar una alerta estructural validaría con ___.

---
**Checklist:** 13 autoevaluaciones en ✅ · codo, DBSCAN y comparativa revisados · [`data/DATOS.md`](data/DATOS.md) leído.
